# Fixed universal attacks evaluated on AnomalyCLIP

This notebook is an independent Kaggle entry point for AnomalyCLIP. It clones the experiment and official model repositories, resolves the exact canonical perturbation archive, uses the manifest's fixed target IDs, and reports clean/adversarial metrics. Enable a GPU and Internet before running all cells.

In [ ]:
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE REPOSITORIES AND INSTALL DEPENDENCIES =====')
WORKING = Path('/kaggle/working')
EXPERIMENT_ROOT = WORKING / 'adversarial-robustness'
ANOMALYCLIP_ROOT = WORKING / 'AnomalyCLIP'
EXPERIMENT_REPO_URL = 'https://github.com/Parsagh05/adversarial-robustness.git'
ANOMALYCLIP_REPO_URL = 'https://github.com/zqhang/AnomalyCLIP.git'
# This is the official AnomalyCLIP revision recorded by the canonical artifacts.
ANOMALYCLIP_COMMIT = '3911738c0867544f545a076ad78f3f11d9ecbfdf'

def clone_or_update(url, destination, commit=None):
    if destination.exists():
        subprocess.run(['git', '-C', str(destination), 'fetch', '--all', '--tags'], check=True)
    else:
        subprocess.run(['git', 'clone', url, str(destination)], check=True)
    if commit:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    else:
        subprocess.run(['git', '-C', str(destination), 'pull', '--ff-only'], check=True)

clone_or_update(EXPERIMENT_REPO_URL, EXPERIMENT_ROOT)
clone_or_update(ANOMALYCLIP_REPO_URL, ANOMALYCLIP_ROOT, ANOMALYCLIP_COMMIT)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(EXPERIMENT_ROOT / 'new_pipeline' / 'requirements.txt')
], check=True)
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))
print('Experiment code:', EXPERIMENT_ROOT / 'new_pipeline')
print('Official AnomalyCLIP:', ANOMALYCLIP_ROOT)

In [ ]:
import shutil
import gdown

from new_pipeline.universal_eval.artifacts import load_manifest

print('===== STEP 2: RESOLVE THE EXACT CANONICAL ARTIFACT ARCHIVE =====')
DRIVE_FILE_ID = '10ZiaDs6u5G_WFVbrd9tt-Ug6Dy_aaFS5'
ARTIFACT_DIR_NAME = 'canonical_clip_universal_attacks_full'

def valid_artifact_root(path):
    return path.is_dir() and (path / 'all_canonical_attack_artifacts.json').is_file()

candidates = [
    EXPERIMENT_ROOT / 'new_pipeline' / 'perturbations' / ARTIFACT_DIR_NAME,
    WORKING / ARTIFACT_DIR_NAME,
]
if Path('/kaggle/input').is_dir():
    candidates.extend(Path('/kaggle/input').rglob(ARTIFACT_DIR_NAME))
ARTIFACTS_ROOT = next((path for path in candidates if valid_artifact_root(path)), None)

if ARTIFACTS_ROOT is None:
    archive_path = WORKING / 'canonical_clip_universal_attacks_full_ARTIFACTS.zip'
    print('Artifacts are not attached as a Kaggle input; downloading the exact Drive file...')
    downloaded = gdown.download(id=DRIVE_FILE_ID, output=str(archive_path), quiet=False)
    if not downloaded or not archive_path.is_file():
        raise RuntimeError(
            'Drive download failed. Attach the ZIP as a private Kaggle dataset and rerun.'
        )
    extract_root = WORKING / 'canonical_artifact_extract'
    extract_root.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(archive_path), str(extract_root))
    extracted = [path for path in extract_root.rglob(ARTIFACT_DIR_NAME) if valid_artifact_root(path)]
    if not extracted and valid_artifact_root(extract_root):
        extracted = [extract_root]
    if len(extracted) != 1:
        raise RuntimeError(f'Expected one extracted artifact root, found: {extracted}')
    ARTIFACTS_ROOT = extracted[0]

artifacts = load_manifest(ARTIFACTS_ROOT)
print('Canonical root:', ARTIFACTS_ROOT)
print('Available conditions:', len(artifacts))
for source, target in sorted({(a.record['source_dataset'], a.record['target_dataset']) for a in artifacts}):
    count = sum(a.record['source_dataset'] == source and a.record['target_dataset'] == target for a in artifacts)
    print(f'  {source} -> {target}: {count}')

In [ ]:
import torch

print('===== STEP 3: RESOLVE DATASETS AND TARGET CHECKPOINTS =====')

def first_existing_directory(paths, label):
    for path in paths:
        if path.is_dir():
            return path
    raise FileNotFoundError(f'{label} was not found. Checked: {paths}')

MVTEC_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection'),
    Path('/kaggle/input/mvtec-ad/mvtec_anomaly_detection'),
], 'MVTec AD')
VISA_ROOT = first_existing_directory([
    Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922'),
    Path('/kaggle/input/visa-ad/VisA_20220922'),
], 'VisA')
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')

# Opposite-dataset checkpoints preserve the original zero-shot protocol.
MVTEC_TARGET_CHECKPOINT = ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale' / 'epoch_15.pth'
VISA_TARGET_CHECKPOINT = ANOMALYCLIP_ROOT / 'checkpoints' / '9_12_4_multiscale_visa' / 'epoch_15.pth'
for checkpoint in (MVTEC_TARGET_CHECKPOINT, VISA_TARGET_CHECKPOINT):
    if not checkpoint.is_file():
        available = sorted((ANOMALYCLIP_ROOT / 'checkpoints').rglob('*.pth'))
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint}. Available: {available}')

MODEL_KWARGS_BY_TARGET = {
    'mvtec': {
        'repository_root': str(ANOMALYCLIP_ROOT),
        'checkpoint_path': str(MVTEC_TARGET_CHECKPOINT),
        'clip_download_root': str(WORKING / 'clip_cache'),
    },
    'visa': {
        'repository_root': str(ANOMALYCLIP_ROOT),
        'checkpoint_path': str(VISA_TARGET_CHECKPOINT),
        'clip_download_root': str(WORKING / 'clip_cache'),
    },
}
print('MVTec:', MVTEC_ROOT)
print('VisA:', VISA_ROOT)
print('MVTec target checkpoint:', MVTEC_TARGET_CHECKPOINT)
print('VisA target checkpoint:', VISA_TARGET_CHECKPOINT)

In [ ]:
from new_pipeline import EvaluationConfig, run_evaluation

print('===== STEP 4: RUN FIXED-ID CLEAN/ADVERSARIAL EVALUATION =====')
FULL_RUN = True
OUTPUT_ROOT = WORKING / (
    'kaggle_new_anomalyclip_full' if FULL_RUN else 'kaggle_new_anomalyclip_check'
)

config = EvaluationConfig(
    artifacts_root=str(ARTIFACTS_ROOT),
    mvtec_root=str(MVTEC_ROOT),
    visa_root=str(VISA_ROOT),
    output_root=str(OUTPUT_ROOT),
    model_name='anomalyclip',
    model_kwargs_by_target=MODEL_KWARGS_BY_TARGET,
    device='cuda',
    batch_size=2,
    metric_size=518,
    anomaly_map_sigma=4.0,
    aupro_fpr_limit=0.30,
    aupro_max_thresholds=200,
    verify_checksums=True,
    save_predictions=True,
    max_conditions=None if FULL_RUN else 1,
    run_notes='Exact canonical Drive artifacts; fixed manifest evaluation IDs.',
)
SUMMARY_PATH = run_evaluation(config)
print('Finished:', SUMMARY_PATH)

In [ ]:
import csv

print('===== STEP 5: PREVIEW SUMMARY =====')
with SUMMARY_PATH.open(newline='', encoding='utf-8') as handle:
    summary_rows = list(csv.DictReader(handle))
columns = [
    'source_dataset', 'target_dataset', 'direction', 'loss_mode',
    'clean_i_auroc', 'adversarial_i_auroc', 'delta_i_auroc',
    'clean_p_auroc', 'adversarial_p_auroc', 'delta_p_auroc',
    'clean_aupro', 'adversarial_aupro', 'delta_aupro',
]
for row in summary_rows:
    print({column: row[column] for column in columns})

In [ ]:
print('===== STEP 6: PACKAGE OUTPUTS =====')
archive = shutil.make_archive(
    str(OUTPUT_ROOT), 'zip', root_dir=OUTPUT_ROOT.parent, base_dir=OUTPUT_ROOT.name
)
print('Packaged results:', archive)